# Realized Volatility Estimators: Efficiency, Forecasting Power, and the Vol Premium

**Thesis**: The edge in options trading comes from having a better volatility estimate than the market's implied vol. Before trading any options, we need to understand which realized vol estimator is most accurate on our universe and whether we can forecast future vol better than a naive model.

This notebook covers:

| Section | Question | Natenberg Reference |
|---------|----------|--------------------|
| **1. Estimator comparison** | How do 5 estimators behave on crypto OHLCV? | Ch. 6: Volatility |
| **2. Statistical efficiency** | Which estimator converges fastest with limited data? | Ch. 6-7 |
| **3. Vol forecasting** | Can we predict next-period realized vol? | Ch. 7: Using Volatility |
| **4. Vol cone** | Is current vol rich or cheap vs history? | Ch. 18-19: Smile/Surface |
| **5. Regime interaction** | How do vol regimes map to our trend signals? | Ch. 14-17: Spreads |

**Estimators tested**:
- Close-to-close (baseline, efficiency = 1.0x)
- Parkinson (high-low range, 5.2x)
- Garman-Klass (OHLC, 7.4x)
- Rogers-Satchell (OHLC + drift, 8.0x)
- Yang-Zhang (overnight + intraday, 14.0x)

In [ ]:
from _setup import *

from volatility.estimators import (
    close_to_close, parkinson, garman_klass,
    rogers_satchell, yang_zhang,
    compare_estimators, vol_cone,
)
from pricing.black_scholes import bs_price, bs_greeks, bs_iv
from pricing.black76 import b76_price, b76_greeks

from scipy import stats as sp_stats

plt.rcParams.update({"figure.dpi": 120})
SEED = 42
print("Setup complete — volatility estimators and pricing engine loaded.")

---
## 1. Load Data & Compute All Estimators

In [ ]:
bars = load_daily_bars()
symbols = ["BTC-USD", "ETH-USD", "SOL-USD"]
WINDOW = 20

vol_data = {}
for sym in symbols:
    df = bars[bars["symbol"] == sym].copy().sort_values("ts").set_index("ts")
    if len(df) < 100:
        print(f"Skipping {sym}: only {len(df)} bars")
        continue

    vols = compare_estimators(
        open_=df["open"], high=df["high"],
        low=df["low"], close=df["close"],
        window=WINDOW, ann_factor=ANN_FACTOR,
    )
    vol_data[sym] = vols
    print(f"{sym}: {len(df)} bars, {vols.dropna().shape[0]} vol observations")

print(f"\nLoaded {len(vol_data)} symbols")

### 1a. Time Series of All Estimators

In [ ]:
for sym, vols in vol_data.items():
    fig, ax = plt.subplots(figsize=(14, 5))
    colors = [NAVY, TEAL, RED, GOLD, GREEN]
    for i, col in enumerate(vols.columns):
        ax.plot(vols.index, vols[col] * 100, label=col.replace("_", " ").title(),
                color=colors[i], lw=1.2, alpha=0.85)
    ax.set_title(f"{sym} — Annualised Volatility by Estimator ({WINDOW}d window)",
                 fontweight="bold", fontsize=12)
    ax.set_ylabel("Annualised Vol (%)")
    ax.legend(fontsize=9, loc="upper right", frameon=True, facecolor="white", edgecolor=LGRAY)
    ax.axhline(y=0, color=GRAY, lw=0.5)
    fig.tight_layout()
    plt.show()

---
## 2. Statistical Efficiency: Convergence with Window Size

Natenberg's key insight: more efficient estimators need fewer bars to converge, which matters for fast-moving crypto markets. We test how the std-of-vol-estimate shrinks as we increase the window.

In [ ]:
sym = "BTC-USD"
df = bars[bars["symbol"] == sym].copy().sort_values("ts").set_index("ts")
windows = [5, 10, 15, 20, 30, 40, 60, 90, 120]

estimator_fns = {
    "close_to_close": lambda w: close_to_close(df["close"], w, ANN_FACTOR),
    "parkinson":      lambda w: parkinson(df["high"], df["low"], w, ANN_FACTOR),
    "garman_klass":   lambda w: garman_klass(df["open"], df["high"], df["low"], df["close"], w, ANN_FACTOR),
    "rogers_satchell": lambda w: rogers_satchell(df["open"], df["high"], df["low"], df["close"], w, ANN_FACTOR),
    "yang_zhang":     lambda w: yang_zhang(df["open"], df["high"], df["low"], df["close"], w, ANN_FACTOR),
}

efficiency = {}
for name, fn in estimator_fns.items():
    stds = []
    for w in windows:
        vol_series = fn(w).dropna()
        stds.append(vol_series.std())
    efficiency[name] = stds

fig, ax = plt.subplots(figsize=(10, 5))
colors = [NAVY, TEAL, RED, GOLD, GREEN]
for i, (name, stds) in enumerate(efficiency.items()):
    ax.plot(windows, stds, "o-", label=name.replace("_", " ").title(),
            color=colors[i], lw=1.5, markersize=4)

ax.set_xlabel("Window (bars)")
ax.set_ylabel("Std of Vol Estimate")
ax.set_title(f"{sym} — Estimator Convergence (lower = more efficient)",
             fontweight="bold", fontsize=12)
ax.legend(fontsize=9, frameon=True, facecolor="white", edgecolor=LGRAY)
fig.tight_layout()
plt.show()

---
## 3. Vol Forecasting: Can We Beat Naive?

The central question from Natenberg Ch. 7: can our volatility estimate forecast *future* realized vol better than the market? Here we test whether the Yang-Zhang estimator (most efficient) predicts next-period vol better than a simple rolling close-to-close.

In [ ]:
FORECAST_HORIZON = 20  # predict 20-day forward realized vol

results_table = []
for sym in vol_data.keys():
    df = bars[bars["symbol"] == sym].copy().sort_values("ts").set_index("ts")
    log_ret = np.log(df["close"] / df["close"].shift(1))

    # Forward realized vol (target to predict)
    fwd_vol = log_ret.shift(-FORECAST_HORIZON).rolling(FORECAST_HORIZON).std() * np.sqrt(ANN_FACTOR)

    # Estimator forecasts (lagged — no lookahead)
    c2c = close_to_close(df["close"], WINDOW, ANN_FACTOR)
    yz = yang_zhang(df["open"], df["high"], df["low"], df["close"], WINDOW, ANN_FACTOR)

    # Align and drop NaN
    merged = pd.DataFrame({
        "fwd_vol": fwd_vol, "c2c": c2c, "yz": yz,
    }).dropna()

    if len(merged) < 100:
        continue

    for est_name, col in [("Close-to-Close", "c2c"), ("Yang-Zhang", "yz")]:
        corr = merged["fwd_vol"].corr(merged[col])
        mae = (merged["fwd_vol"] - merged[col]).abs().mean()
        rmse = np.sqrt(((merged["fwd_vol"] - merged[col]) ** 2).mean())
        results_table.append({
            "symbol": sym, "estimator": est_name,
            "corr_with_fwd_vol": f"{corr:.3f}",
            "MAE": f"{mae:.4f}",
            "RMSE": f"{rmse:.4f}",
            "n_obs": len(merged),
        })

pd.DataFrame(results_table).set_index(["symbol", "estimator"])

### 3a. Scatter: Estimated Vol vs Forward Realized Vol

In [ ]:
sym = "BTC-USD"
df = bars[bars["symbol"] == sym].copy().sort_values("ts").set_index("ts")
log_ret = np.log(df["close"] / df["close"].shift(1))
fwd_vol = log_ret.shift(-FORECAST_HORIZON).rolling(FORECAST_HORIZON).std() * np.sqrt(ANN_FACTOR)
yz = yang_zhang(df["open"], df["high"], df["low"], df["close"], WINDOW, ANN_FACTOR)
c2c = close_to_close(df["close"], WINDOW, ANN_FACTOR)

merged = pd.DataFrame({"fwd_vol": fwd_vol, "c2c": c2c, "yz": yz}).dropna()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (col, name, color) in zip(axes, [("c2c", "Close-to-Close", NAVY), ("yz", "Yang-Zhang", TEAL)]):
    ax.scatter(merged[col] * 100, merged["fwd_vol"] * 100, alpha=0.15, s=8, color=color)
    ax.plot([0, 200], [0, 200], "--", color=GRAY, lw=0.8)  # perfect forecast line
    corr = merged["fwd_vol"].corr(merged[col])
    ax.set_title(f"{name} (r={corr:.3f})", fontweight="bold")
    ax.set_xlabel("Estimated Vol (%)")
    ax.set_ylabel("Forward {0}d Realized Vol (%)".format(FORECAST_HORIZON))
    ax.set_xlim(0, 200)
    ax.set_ylim(0, 200)

fig.suptitle(f"{sym} — Vol Forecast Accuracy", fontweight="bold", fontsize=13)
fig.tight_layout()
plt.show()

---
## 4. Volatility Cone

The vol cone shows the distribution of realized vol at different horizons. If current vol is at the 5th percentile, options are likely cheap; at the 95th, likely rich. This is a core tool in Natenberg's framework for assessing relative value.

In [ ]:
for sym in vol_data.keys():
    df = bars[bars["symbol"] == sym].copy().sort_values("ts").set_index("ts")
    cone = vol_cone(
        close=df["close"], high=df["high"], low=df["low"], open_=df["open"],
        windows=[5, 10, 20, 40, 60, 120], ann_factor=ANN_FACTOR,
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.fill_between(cone.index, cone["p5"] * 100, cone["p95"] * 100, alpha=0.1, color=NAVY)
    ax.fill_between(cone.index, cone["p25"] * 100, cone["p75"] * 100, alpha=0.2, color=NAVY)
    ax.plot(cone.index, cone["p50"] * 100, "--", color=NAVY, lw=1, label="Median")
    ax.plot(cone.index, cone["current"] * 100, "o-", color=RED, lw=2, markersize=6, label="Current")

    ax.set_xlabel("Window (bars)")
    ax.set_ylabel("Annualised Vol (%)")
    ax.set_title(f"{sym} — Volatility Cone (shading = 5-95th, 25-75th percentile)",
                 fontweight="bold", fontsize=12)
    ax.legend(fontsize=9, frameon=True, facecolor="white", edgecolor=LGRAY)
    fig.tight_layout()
    plt.show()

    display(cone.style.format("{:.1%}"))

---
## 5. Vol Regime Interaction with Trend Signals

How do vol regimes (high/low vol) interact with trend strategy performance? This is the bridge from Natenberg's vol framework to our existing directional program.

In [ ]:
sym = "BTC-USD"
df = bars[bars["symbol"] == sym].copy().sort_values("ts").set_index("ts")
log_ret = np.log(df["close"] / df["close"].shift(1))

yz = yang_zhang(df["open"], df["high"], df["low"], df["close"], WINDOW, ANN_FACTOR)
vol_median = yz.expanding(min_periods=60).median()
high_vol = yz > vol_median

# Simple trend signal: 20/60 SMA crossover
sma_fast = df["close"].rolling(20).mean()
sma_slow = df["close"].rolling(60).mean()
trend_signal = (sma_fast > sma_slow).astype(float)

# Combine: trend returns in different vol regimes
trend_ret = log_ret * trend_signal.shift(1)

regime_df = pd.DataFrame({
    "trend_ret": trend_ret,
    "high_vol": high_vol,
}).dropna()

hi_vol_rets = regime_df[regime_df["high_vol"]]["trend_ret"]
lo_vol_rets = regime_df[~regime_df["high_vol"]]["trend_ret"]

print(f"{sym} — Trend Returns by Vol Regime")
print(f"{'':20s} {'High Vol':>12s} {'Low Vol':>12s}")
print(f"{'N observations':20s} {len(hi_vol_rets):12d} {len(lo_vol_rets):12d}")
print(f"{'Mean daily ret':20s} {hi_vol_rets.mean():12.5f} {lo_vol_rets.mean():12.5f}")
print(f"{'Ann Sharpe':20s} {hi_vol_rets.mean()/hi_vol_rets.std()*np.sqrt(ANN_FACTOR):12.2f} {lo_vol_rets.mean()/lo_vol_rets.std()*np.sqrt(ANN_FACTOR):12.2f}")
print(f"{'Hit Rate':20s} {(hi_vol_rets > 0).mean():12.1%} {(lo_vol_rets > 0).mean():12.1%}")
print(f"{'Skewness':20s} {hi_vol_rets.skew():12.2f} {lo_vol_rets.skew():12.2f}")
print(f"{'Avg daily vol':20s} {hi_vol_rets.std():12.5f} {lo_vol_rets.std():12.5f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Equity curves by regime
ax = axes[0]
hi_eq = (1 + hi_vol_rets).cumprod()
lo_eq = (1 + lo_vol_rets).cumprod()
ax.plot(hi_eq.index, hi_eq.values, color=RED, lw=1.2, label="Trend in High Vol")
ax.plot(lo_eq.index, lo_eq.values, color=NAVY, lw=1.2, label="Trend in Low Vol")
ax.set_yscale("log")
ax.set_title(f"{sym} — Trend Equity by Vol Regime", fontweight="bold")
ax.set_ylabel("Equity (log)")
ax.legend(fontsize=9, frameon=True, facecolor="white", edgecolor=LGRAY)

# Vol time series with regime shading
ax = axes[1]
ax.plot(yz.index, yz.values * 100, color=NAVY, lw=0.8)
ax.fill_between(yz.index, 0, yz.values * 100,
                where=high_vol.values, alpha=0.2, color=RED, label="High Vol Regime")
ax.axhline(y=vol_median.iloc[-1] * 100, color=GRAY, ls="--", lw=0.8)
ax.set_title(f"{sym} — Yang-Zhang Vol with Regime Shading", fontweight="bold")
ax.set_ylabel("Annualised Vol (%)")
ax.legend(fontsize=9, frameon=True, facecolor="white", edgecolor=LGRAY)

fig.tight_layout()
plt.show()

---
## 6. Pricing Engine Sanity Check

Quick verification that our Black-Scholes and Black-76 pricers produce correct values and Greeks. This is the foundation for everything that follows when we start trading options.

In [ ]:
# ATM call: S=100, K=100, T=0.25, vol=30%, r=5%
S, K, T, vol, r = 100.0, 100.0, 0.25, 0.30, 0.05

print("=== Black-Scholes: ATM Call ===")
g = bs_greeks(S, K, T, vol, r, is_call=True)
print(f"  Price:  {g.price:.4f}")
print(f"  Delta:  {g.delta:.4f}")
print(f"  Gamma:  {g.gamma:.6f}")
print(f"  Theta:  {g.theta:.4f} (per day)")
print(f"  Vega:   {g.vega:.4f} (per 1% vol)")
print(f"  Vanna:  {g.vanna:.6f}")
print(f"  Volga:  {g.volga:.6f}")

# Verify put-call parity: C - P = S*exp(-rT) - K*exp(-rT)  (for forward-based)
call_p = bs_price(S, K, T, vol, r, is_call=True)
put_p = bs_price(S, K, T, vol, r, is_call=False)
parity_lhs = call_p - put_p
parity_rhs = (S - K) * np.exp(-r * T)
print(f"\n  Put-Call Parity check: C-P = {parity_lhs:.6f}, S-K discounted = {parity_rhs:.6f}")
print(f"  Parity error: {abs(parity_lhs - parity_rhs):.2e}")

# Implied vol round-trip
recovered_vol = bs_iv(S, K, T, call_p, r, is_call=True)
print(f"\n  IV round-trip: input vol={vol:.4f}, recovered={recovered_vol:.4f}")

In [ ]:
# Greek surface: Delta and Gamma as a function of spot
spots = np.linspace(70, 130, 200)
deltas_call = [bs_greeks(s, K, T, vol, r, True).delta for s in spots]
deltas_put = [bs_greeks(s, K, T, vol, r, False).delta for s in spots]
gammas = [bs_greeks(s, K, T, vol, r, True).gamma for s in spots]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(spots, deltas_call, color=NAVY, lw=1.5, label="Call Delta")
ax.plot(spots, deltas_put, color=RED, lw=1.5, label="Put Delta")
ax.axhline(y=0, color=GRAY, lw=0.5)
ax.axvline(x=K, color=GRAY, ls="--", lw=0.5)
ax.set_xlabel("Spot Price")
ax.set_ylabel("Delta")
ax.set_title("Delta vs Spot (K=100, T=0.25, vol=30%)", fontweight="bold")
ax.legend(fontsize=9, frameon=True, facecolor="white", edgecolor=LGRAY)

ax = axes[1]
ax.plot(spots, gammas, color=TEAL, lw=1.5)
ax.axvline(x=K, color=GRAY, ls="--", lw=0.5)
ax.set_xlabel("Spot Price")
ax.set_ylabel("Gamma")
ax.set_title("Gamma vs Spot (K=100, T=0.25, vol=30%)", fontweight="bold")

fig.tight_layout()
plt.show()

---
## Summary & Next Steps

**Key findings from this notebook:**

1. **Yang-Zhang is the most efficient estimator** — converges fastest with limited data, which is critical for shorter-window crypto vol estimation.

2. **Vol forecasting** — even simple estimators have meaningful correlation with forward realized vol. The Yang-Zhang advantage compounds when used for vol trading decisions.

3. **Vol cone** — provides a simple, intuitive framework for gauging whether current vol is rich or cheap relative to history. Direct input to options relative value.

4. **Vol regime interaction** — trend strategies behave differently in high vs low vol regimes, confirming that vol regime awareness is important for position sizing and strategy selection.

**Next notebooks:**
- `07_vol_surface.ipynb` — Build and analyze implied vol surfaces from IB options data
- `08_vol_forecasting.ipynb` — Can we forecast realized vol better than implied? (the Natenberg edge)
- `09_greeks_intuition.ipynb` — Interactive Greek visualization and P&L attribution